# Reproducible Prompt Engineering Experiments

**Project:** Bilingual question-type classification with prompt sensitivity and consistency  
**Task labels:** `NUMBER`, `LOCATION`, `PERSON`, `DESCRIPTION`, `ENTITY`, `ABBREVIATION`

This notebook reproduces the complete experiment sequence recorded in the research log.

## Preliminary Falcon3 pilots

- **Pilot_01:** tiny 12-sample pipeline/debugging pilot
- **Pilot_02:** expanded 120-sample pre-cleaning dataset pilot

## Falcon3 prompt/dataset progression

- **Pilot_03:** cleaned dataset + simple prompts
- **Pilot_04:** label definitions
- **Pilot_05:** definitions + few-shot examples
- **Pilot_06:** definitions + few-shot examples + `DESCRIPTION`/`ENTITY` disambiguation
- **Pilot_07:** unseen validation using the frozen final prompt

## Final cross-model validation

Open models:

- `tiiuae/Falcon3-7B-Instruct`
- `humain-ai/ALLaM-7B-Instruct-preview`
- `FreedomIntelligence/AceGPT-v2-8B-Chat`

Closed model:

- `gemini-2.5-flash`

All outputs are saved as CSV files under `results/`.

## Before running

1. In Colab, go to **Runtime → Change runtime type → T4 GPU** if running Hugging Face models.
2. For Hugging Face models, optionally add `HF_TOKEN` in Colab Secrets.
3. For Gemini, add your API key in Colab Secrets as exactly `GEMINI_API_KEY`.
4. Do not paste API keys directly into the notebook.
5. Choose the run mode in the configuration cell below.

Recommended current mode: `gemini_only`, because the Hugging Face results already exist and only Gemini needs to be added.

## 0. Colab setup

Use a GPU runtime for Hugging Face models:

`Runtime → Change runtime type → T4 GPU`

Gemini does not require GPU, but keeping a GPU runtime is fine.

The Hugging Face models are public, but adding a Hugging Face token in Colab Secrets as `HF_TOKEN` can reduce rate-limit or download issues.

For Gemini, add a secret named exactly:

`GEMINI_API_KEY`

Since old installs caused dependency conflicts, before running this corrected notebook use:

`Runtime → Disconnect and delete runtime`

Then reconnect and run from the top.

In [ ]:
# Install dependencies.
# Colab-safe versions for this project.
# Do not force google-auth because Colab manages it internally.

%pip install -q \
    "pandas==2.2.2" \
    "google-genai==1.66.0" \
    "bitsandbytes>=0.46.1" \
    "accelerate>=0.34.0" \
    "transformers>=4.46.0" \
    "scikit-learn" \
    "tqdm"

In [ ]:
# Verify environment

import os
import pandas as pd

print("pandas:", pd.__version__)

try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    print("torch:", torch.__version__)
except Exception as e:
    print("torch not available or not needed for Gemini-only mode:", e)

try:
    import transformers
    import accelerate
    import bitsandbytes as bnb

    print("transformers:", transformers.__version__)
    print("accelerate:", accelerate.__version__)
    print("bitsandbytes:", bnb.__version__)
except Exception as e:
    print("HF model packages not fully available. This is okay for gemini_only mode.")
    print(e)

try:
    from google import genai
    print("google-genai import works.")
except Exception as e:
    print("google-genai import failed:", e)

## 1. Choose run mode

Choose one mode before running the notebook.

Recommended current mode:

- `local_results_only`: verify local datasets, prompts, metrics, and checked-in result CSVs without calling external models/APIs.
- `gemini_only`: rerun only Gemini closed-model validation when `GEMINI_API_KEY` is available.

Other modes:

- `hf_cross_model`: run Falcon3 + ALLaM + AceGPT on Pilot_07.
- `falcon_progression`: run Pilot_01 through Pilot_07 on Falcon3.
- `full`: run everything.
- `smoke_test`: run tiny model checks only.
- `local_results_only`: no external model calls; use the existing repository CSV files.

For final paper submission, use `gemini_only` now because the Hugging Face CSV results already exist.

In [ ]:
# =========================
# RUN MODE CONFIGURATION
# =========================

RUN_MODE = "local_results_only"

# Options:
# "gemini_only"
# "hf_cross_model"
# "falcon_progression"
# "full"
# "smoke_test"
# "local_results_only"

QUICK_TEST = RUN_MODE == "smoke_test"
MAX_SAMPLES_PER_LANGUAGE = 2 if QUICK_TEST else None

RUN_FALCON_PROMPT_PROGRESSION = RUN_MODE in {"falcon_progression", "full", "smoke_test"}
RUN_CROSS_MODEL_VALIDATION = RUN_MODE in {"hf_cross_model", "full", "smoke_test"}
RUN_GEMINI_CLOSED_MODEL_VALIDATION = RUN_MODE in {"gemini_only", "full", "smoke_test"}

print("RUN_MODE:", RUN_MODE)
print("RUN_FALCON_PROMPT_PROGRESSION:", RUN_FALCON_PROMPT_PROGRESSION)
print("RUN_CROSS_MODEL_VALIDATION:", RUN_CROSS_MODEL_VALIDATION)
print("RUN_GEMINI_CLOSED_MODEL_VALIDATION:", RUN_GEMINI_CLOSED_MODEL_VALIDATION)
print("QUICK_TEST:", QUICK_TEST)

## 1. Global configuration

In [ ]:
import os
import gc
import io
import json
import math
import random
import shutil
import re
import time
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score
from IPython.display import display

try:
    import torch
except Exception:
    torch = None

# =========================
# Reproducibility controls
# =========================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

if torch is not None:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

# =========================
# Output directory
# =========================

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# Models
# =========================

FALCON_MODEL_ID = "tiiuae/Falcon3-7B-Instruct"
ALLAM_MODEL_ID = "humain-ai/ALLaM-7B-Instruct-preview"
ACEGPT_MODEL_ID = "FreedomIntelligence/AceGPT-v2-8B-Chat"
GEMINI_MODEL_ID = "gemini-2.5-flash"

CROSS_MODEL_RUN_ORDER = [
    ("falcon3", FALCON_MODEL_ID, "General baseline"),
    ("allam", ALLAM_MODEL_ID, "Arabic-focused"),
    ("acegpt", ACEGPT_MODEL_ID, "Arabic-focused"),
]

LABELS = [
    "NUMBER",
    "LOCATION",
    "PERSON",
    "DESCRIPTION",
    "ENTITY",
    "ABBREVIATION",
]

print("Results directory:", RESULTS_DIR.resolve())
print("Run mode:", RUN_MODE)

In [ ]:
# Check Colab secrets / environment variables without revealing them.

def has_secret(name: str) -> bool:
    value = os.environ.get(name)

    try:
        from google.colab import userdata
        value = userdata.get(name) or value
    except Exception:
        pass

    return bool(value)


print("HF_TOKEN available:", has_secret("HF_TOKEN"))
print("GEMINI_API_KEY available:", has_secret("GEMINI_API_KEY"))

if RUN_GEMINI_CLOSED_MODEL_VALIDATION and not has_secret("GEMINI_API_KEY"):
    print("\nWARNING: RUN_GEMINI_CLOSED_MODEL_VALIDATION=True but GEMINI_API_KEY is missing.")
    print("Add it in Colab Secrets before running the Gemini section.")

## 2. Datasets

This notebook embeds all datasets needed for reproducibility:

- `pilot01_df`: tiny 12-sample starter pilot used to debug the pipeline.
- `pilot02_df`: expanded 120-sample pre-cleaning dataset.
- `development_df`: cleaned 120-sample development dataset used for Pilot_03--Pilot_06.
- `validation_df`: unseen 120-sample validation dataset used for Pilot_07 and cross-model validation.


In [ ]:
# =========================
# Pilot_01 tiny starter dataset used for pipeline debugging
# =========================

PILOT01_TSV = """id	label	language	question
en1	PERSON	English	Who invented the telephone?
en2	LOCATION	English	Where is the Eiffel Tower located?
en3	NUMBER	English	How many planets are in the solar system?
en4	ABBREVIATION	English	What does CPU stand for?
en5	DESCRIPTION	English	What is a volcano?
en6	ENTITY	English	What is the name of the largest ocean?
ar1	PERSON	Arabic	من اخترع الهاتف؟
ar2	LOCATION	Arabic	أين يقع برج إيفل؟
ar3	NUMBER	Arabic	كم عدد كواكب المجموعة الشمسية؟
ar4	ABBREVIATION	Arabic	ماذا يعني اختصار CPU؟
ar5	DESCRIPTION	Arabic	ما هو البركان؟
ar6	ENTITY	Arabic	ما اسم أكبر محيط في العالم؟"""


# =========================
# Pilot_02 expanded pre-cleaning dataset
# =========================

PILOT02_TSV = """id	label	language	question
1	NUMBER	English	How many planets are in the Solar System?
2	NUMBER	English	What year did the first iPhone launch?
3	NUMBER	English	How many sides does a hexagon have?
4	NUMBER	English	How many minutes are in two hours?
5	NUMBER	English	What is the boiling point of water in Celsius?
6	NUMBER	English	How many players are on a standard soccer team?
7	NUMBER	English	What is the speed of light in kilometers per second?
8	NUMBER	English	How many bones are in the adult human body?
9	NUMBER	English	What is the area of a rectangle that is 8 meters long and 5 meters wide?
10	NUMBER	English	How many days are there in a leap year?
11	NUMBER	Arabic	كم عدد الكواكب في النظام الشمسي؟
12	NUMBER	Arabic	في أي سنة انطلقت أول دورة للألعاب الأولمبية الحديثة؟
13	NUMBER	Arabic	كم ضلعًا للمثلث؟
14	NUMBER	Arabic	ما عدد الدقائق في ثلاث ساعات؟
15	NUMBER	Arabic	كم عدد أيام الأسبوع؟
16	NUMBER	Arabic	ما درجة غليان الماء بالسيلسيوس؟
17	NUMBER	Arabic	كم لاعبًا يوجد في فريق كرة القدم داخل الملعب؟
18	NUMBER	Arabic	كم عدد الحروف في الأبجدية الإنجليزية؟
19	NUMBER	Arabic	ما نصف العدد 100؟
20	NUMBER	Arabic	كم شهرًا في السنة الميلادية؟
21	LOCATION	English	Where is the Eiffel Tower located?
22	LOCATION	English	What country is Tokyo in?
23	LOCATION	English	Where is the Sahara Desert located?
24	LOCATION	English	Which city is the capital of Canada?
25	LOCATION	English	Where is Mount Everest located?
26	LOCATION	English	What continent is Brazil located in?
27	LOCATION	English	Where is the Great Wall of China?
28	LOCATION	English	Which country contains the city of Barcelona?
29	LOCATION	English	Where is the Amazon River found?
30	LOCATION	English	What is the capital city of Tunisia?
31	LOCATION	Arabic	أين يقع برج إيفل؟
32	LOCATION	Arabic	في أي دولة تقع مدينة طوكيو؟
33	LOCATION	Arabic	أين تقع الصحراء الكبرى؟
34	LOCATION	Arabic	ما المدينة التي تعد عاصمة كندا؟
35	LOCATION	Arabic	أين يقع جبل إيفرست؟
36	LOCATION	Arabic	في أي قارة تقع البرازيل؟
37	LOCATION	Arabic	أين يوجد سور الصين العظيم؟
38	LOCATION	Arabic	في أي دولة تقع مدينة برشلونة؟
39	LOCATION	Arabic	أين يوجد نهر الأمازون؟
40	LOCATION	Arabic	ما عاصمة تونس؟
41	PERSON	English	Who wrote Romeo and Juliet?
42	PERSON	English	Who discovered gravity according to the famous apple story?
43	PERSON	English	Who was the first person to walk on the Moon?
44	PERSON	English	Who painted the Mona Lisa?
45	PERSON	English	Who is the founder of Microsoft?
46	PERSON	English	Who invented the telephone?
47	PERSON	English	Who was Albert Einstein?
48	PERSON	English	Who is the main character in Harry Potter?
49	PERSON	English	Who was the first president of the United States?
50	PERSON	English	Who composed the Ninth Symphony?
51	PERSON	Arabic	من كتب مسرحية روميو وجولييت؟
52	PERSON	Arabic	من اكتشف الجاذبية حسب قصة التفاحة المشهورة؟
53	PERSON	Arabic	من كان أول إنسان يمشي على سطح القمر؟
54	PERSON	Arabic	من رسم لوحة الموناليزا؟
55	PERSON	Arabic	من هو مؤسس شركة مايكروسوفت؟
56	PERSON	Arabic	من اخترع الهاتف؟
57	PERSON	Arabic	من هو ألبرت أينشتاين؟
58	PERSON	Arabic	من هي الشخصية الرئيسية في سلسلة هاري بوتر؟
59	PERSON	Arabic	من كان أول رئيس للولايات المتحدة؟
60	PERSON	Arabic	من ألّف السيمفونية التاسعة؟
61	DESCRIPTION	English	What is photosynthesis?
62	DESCRIPTION	English	What is artificial intelligence?
63	DESCRIPTION	English	What does democracy mean?
64	DESCRIPTION	English	What is climate change?
65	DESCRIPTION	English	What is the purpose of a database?
66	DESCRIPTION	English	What is machine learning?
67	DESCRIPTION	English	What is a black hole?
68	DESCRIPTION	English	What does inflation mean in economics?
69	DESCRIPTION	English	What is renewable energy?
70	DESCRIPTION	English	What is the function of the heart?
71	DESCRIPTION	Arabic	ما هو التمثيل الضوئي؟
72	DESCRIPTION	Arabic	ما هو الذكاء الاصطناعي؟
73	DESCRIPTION	Arabic	ماذا تعني الديمقراطية؟
74	DESCRIPTION	Arabic	ما هو تغير المناخ؟
75	DESCRIPTION	Arabic	ما وظيفة قاعدة البيانات؟
76	DESCRIPTION	Arabic	ما هو تعلم الآلة؟
77	DESCRIPTION	Arabic	ما هو الثقب الأسود؟
78	DESCRIPTION	Arabic	ماذا يعني التضخم في الاقتصاد؟
79	DESCRIPTION	Arabic	ما هي الطاقة المتجددة؟
80	DESCRIPTION	Arabic	ما وظيفة القلب؟
81	ENTITY	English	What is the largest ocean on Earth?
82	ENTITY	English	What device is used to measure temperature?
83	ENTITY	English	What language is mainly spoken in Brazil?
84	ENTITY	English	What is the chemical symbol for gold?
85	ENTITY	English	What animal is known as the king of the jungle?
86	ENTITY	English	What instrument has black and white keys?
87	ENTITY	English	What planet is known as the Red Planet?
88	ENTITY	English	What currency is used in Japan?
89	ENTITY	English	What programming language is commonly used for Android apps?
90	ENTITY	English	What is the main gas humans need to breathe?
91	ENTITY	Arabic	ما هو أكبر محيط على الأرض؟
92	ENTITY	Arabic	ما الجهاز المستخدم لقياس درجة الحرارة؟
93	ENTITY	Arabic	ما اللغة الرئيسية المستخدمة في البرازيل؟
94	ENTITY	Arabic	ما الرمز الكيميائي للذهب؟
95	ENTITY	Arabic	ما الحيوان المعروف بلقب ملك الغابة؟
96	ENTITY	Arabic	ما الآلة الموسيقية التي تحتوي على مفاتيح سوداء وبيضاء؟
97	ENTITY	Arabic	ما الكوكب المعروف بالكوكب الأحمر؟
98	ENTITY	Arabic	ما العملة المستخدمة في اليابان؟
99	ENTITY	Arabic	ما لغة البرمجة المستخدمة عادة لتطوير تطبيقات أندرويد؟
100	ENTITY	Arabic	ما الغاز الأساسي الذي يحتاجه الإنسان للتنفس؟
101	ABBREVIATION	English	What does NASA stand for?
102	ABBREVIATION	English	What does CPU mean?
103	ABBREVIATION	English	What does HTML stand for?
104	ABBREVIATION	English	What does URL mean?
105	ABBREVIATION	English	What does AI stand for?
106	ABBREVIATION	English	What does GPS stand for?
107	ABBREVIATION	English	What does Wi-Fi stand for?
108	ABBREVIATION	English	What does USB mean?
109	ABBREVIATION	English	What does PDF stand for?
110	ABBREVIATION	English	What does RAM stand for?
111	ABBREVIATION	Arabic	ماذا يعني اختصار NASA؟
112	ABBREVIATION	Arabic	ماذا يعني اختصار CPU؟
113	ABBREVIATION	Arabic	ماذا يعني اختصار HTML؟
114	ABBREVIATION	Arabic	ماذا يعني اختصار URL؟
115	ABBREVIATION	Arabic	ماذا يعني اختصار AI؟
116	ABBREVIATION	Arabic	ماذا يعني اختصار GPS؟
117	ABBREVIATION	Arabic	ماذا يعني اختصار Wi-Fi؟
118	ABBREVIATION	Arabic	ماذا يعني اختصار USB؟
119	ABBREVIATION	Arabic	ماذا يعني اختصار PDF؟
120	ABBREVIATION	Arabic	ماذا يعني اختصار RAM؟"""


# =========================
# Development dataset used for Pilot_03 to Pilot_06
# =========================

DEVELOPMENT_TSV = """id	label	language	question
1	NUMBER	English	How many continents are there on Earth?
2	NUMBER	English	In what year did World War II end?
3	NUMBER	English	How many sides does a pentagon have?
4	NUMBER	English	What is 15 multiplied by 4?
5	NUMBER	English	How many minutes are in one hour?
6	NUMBER	English	What percentage is 25 out of 100?
7	NUMBER	English	What is the freezing point of water in Celsius?
8	NUMBER	English	How many days are in a leap year?
9	NUMBER	English	What is the square root of 81?
10	NUMBER	English	How many centimeters are in one meter?
11	NUMBER	Arabic	كم عدد القارات على سطح الأرض؟
12	NUMBER	Arabic	في أي سنة انتهت الحرب العالمية الثانية؟
13	NUMBER	Arabic	كم ضلعًا للشكل الخماسي؟
14	NUMBER	Arabic	ما ناتج ضرب 15 في 4؟
15	NUMBER	Arabic	كم دقيقة في الساعة الواحدة؟
16	NUMBER	Arabic	ما النسبة المئوية للعدد 25 من 100؟
17	NUMBER	Arabic	ما درجة تجمد الماء بالسيلسيوس؟
18	NUMBER	Arabic	كم يومًا في السنة الكبيسة؟
19	NUMBER	Arabic	ما الجذر التربيعي للعدد 81؟
20	NUMBER	Arabic	كم سنتيمترًا في المتر الواحد؟
21	LOCATION	English	Where is the Eiffel Tower located?
22	LOCATION	English	Which country is Cairo in?
23	LOCATION	English	What city is the capital of Japan?
24	LOCATION	English	On which continent is Kenya located?
25	LOCATION	English	Where is the Nile River located?
26	LOCATION	English	Which desert covers much of North Africa?
27	LOCATION	English	Which country contains Machu Picchu?
28	LOCATION	English	In which city is the Colosseum located?
29	LOCATION	English	Which mountain range separates Europe and Asia?
30	LOCATION	English	Where is the Great Barrier Reef located?
31	LOCATION	Arabic	أين يقع برج إيفل؟
32	LOCATION	Arabic	في أي دولة تقع القاهرة؟
33	LOCATION	Arabic	ما المدينة التي تعد عاصمة اليابان؟
34	LOCATION	Arabic	في أي قارة تقع كينيا؟
35	LOCATION	Arabic	أين يقع نهر النيل؟
36	LOCATION	Arabic	ما الصحراء التي تغطي جزءًا كبيرًا من شمال إفريقيا؟
37	LOCATION	Arabic	في أي دولة تقع ماتشو بيتشو؟
38	LOCATION	Arabic	في أي مدينة يقع الكولوسيوم؟
39	LOCATION	Arabic	ما السلسلة الجبلية التي تفصل بين أوروبا وآسيا؟
40	LOCATION	Arabic	أين يقع الحاجز المرجاني العظيم؟
41	PERSON	English	Who painted the Mona Lisa?
42	PERSON	English	Who wrote Hamlet?
43	PERSON	English	Who invented the telephone?
44	PERSON	English	Who was the first person to walk on the Moon?
45	PERSON	English	Who founded Microsoft with Paul Allen?
46	PERSON	English	Who was the first president of the United States?
47	PERSON	English	Who developed the theory of relativity?
48	PERSON	English	Who is the main character in the Harry Potter books?
49	PERSON	English	Who composed The Four Seasons?
50	PERSON	English	Who is known as the founder of modern nursing?
51	PERSON	Arabic	من رسم لوحة الموناليزا؟
52	PERSON	Arabic	من كتب مسرحية هاملت؟
53	PERSON	Arabic	من اخترع الهاتف؟
54	PERSON	Arabic	من كان أول شخص يمشي على سطح القمر؟
55	PERSON	Arabic	من أسس شركة مايكروسوفت مع بول ألين؟
56	PERSON	Arabic	من كان أول رئيس للولايات المتحدة؟
57	PERSON	Arabic	من طور نظرية النسبية؟
58	PERSON	Arabic	من هي الشخصية الرئيسية في كتب هاري بوتر؟
59	PERSON	Arabic	من ألّف مقطوعة الفصول الأربعة؟
60	PERSON	Arabic	من تُعرف بأنها مؤسسة التمريض الحديث؟
61	DESCRIPTION	English	What is photosynthesis?
62	DESCRIPTION	English	What does democracy mean?
63	DESCRIPTION	English	What is machine learning?
64	DESCRIPTION	English	What is the function of the heart?
65	DESCRIPTION	English	What is renewable energy?
66	DESCRIPTION	English	What does inflation mean in economics?
67	DESCRIPTION	English	What is a black hole?
68	DESCRIPTION	English	What is a database used for?
69	DESCRIPTION	English	What is climate change?
70	DESCRIPTION	English	What is cybersecurity?
71	DESCRIPTION	Arabic	ما هو التمثيل الضوئي؟
72	DESCRIPTION	Arabic	ماذا تعني الديمقراطية؟
73	DESCRIPTION	Arabic	ما هو تعلم الآلة؟
74	DESCRIPTION	Arabic	ما وظيفة القلب؟
75	DESCRIPTION	Arabic	ما هي الطاقة المتجددة؟
76	DESCRIPTION	Arabic	ماذا يعني التضخم في الاقتصاد؟
77	DESCRIPTION	Arabic	ما هو الثقب الأسود؟
78	DESCRIPTION	Arabic	ما استخدام قاعدة البيانات؟
79	DESCRIPTION	Arabic	ما هو تغير المناخ؟
80	DESCRIPTION	Arabic	ما هو الأمن السيبراني؟
81	ENTITY	English	What device is used to measure temperature?
82	ENTITY	English	What animal is known as the king of the jungle?
83	ENTITY	English	What planet is known as the Red Planet?
84	ENTITY	English	What language is mainly spoken in Brazil?
85	ENTITY	English	What currency is used in Japan?
86	ENTITY	English	What musical instrument has black and white keys?
87	ENTITY	English	What gas do humans need to breathe?
88	ENTITY	English	What software application is commonly used to create spreadsheets?
89	ENTITY	English	What element has the atomic number 79?
90	ENTITY	English	What metal is liquid at room temperature?
91	ENTITY	Arabic	ما الجهاز المستخدم لقياس درجة الحرارة؟
92	ENTITY	Arabic	ما الحيوان المعروف بلقب ملك الغابة؟
93	ENTITY	Arabic	ما الكوكب المعروف بالكوكب الأحمر؟
94	ENTITY	Arabic	ما اللغة الرئيسية المستخدمة في البرازيل؟
95	ENTITY	Arabic	ما العملة المستخدمة في اليابان؟
96	ENTITY	Arabic	ما الآلة الموسيقية التي تحتوي على مفاتيح سوداء وبيضاء؟
97	ENTITY	Arabic	ما الغاز الذي يحتاجه الإنسان للتنفس؟
98	ENTITY	Arabic	ما التطبيق المستخدم عادة لإنشاء الجداول الإلكترونية؟
99	ENTITY	Arabic	ما العنصر الذي عدده الذري 79؟
100	ENTITY	Arabic	ما المعدن الذي يكون سائلاً في درجة حرارة الغرفة؟
101	ABBREVIATION	English	What does NASA stand for?
102	ABBREVIATION	English	What does CPU stand for?
103	ABBREVIATION	English	What does HTML stand for?
104	ABBREVIATION	English	What does URL stand for?
105	ABBREVIATION	English	What does AI stand for?
106	ABBREVIATION	English	What does GPS stand for?
107	ABBREVIATION	English	What does USB stand for?
108	ABBREVIATION	English	What does PDF stand for?
109	ABBREVIATION	English	What does RAM stand for?
110	ABBREVIATION	English	What does DNA stand for?
111	ABBREVIATION	Arabic	ماذا يعني اختصار NASA؟
112	ABBREVIATION	Arabic	ماذا يعني اختصار CPU؟
113	ABBREVIATION	Arabic	ماذا يعني اختصار HTML؟
114	ABBREVIATION	Arabic	ماذا يعني اختصار URL؟
115	ABBREVIATION	Arabic	ماذا يعني اختصار AI؟
116	ABBREVIATION	Arabic	ماذا يعني اختصار GPS؟
117	ABBREVIATION	Arabic	ماذا يعني اختصار USB؟
118	ABBREVIATION	Arabic	ماذا يعني اختصار PDF؟
119	ABBREVIATION	Arabic	ماذا يعني اختصار RAM؟
120	ABBREVIATION	Arabic	ماذا يعني اختصار DNA؟
"""

# =========================
# Unseen validation dataset used for Pilot_07 and cross-model validation
# =========================

VALIDATION_TSV = """id	label	language	question
1	NUMBER	English	How many wheels does a standard bicycle have?
2	NUMBER	English	In what year did the Titanic sink?
3	NUMBER	English	How many hours are in a full day?
4	NUMBER	English	What is 12 divided by 3?
5	NUMBER	English	How many degrees are in a right angle?
6	NUMBER	English	What is 20% of 200?
7	NUMBER	English	How many letters are in the English alphabet?
8	NUMBER	English	What is the value of 7 squared?
9	NUMBER	English	How many months have 31 days?
10	NUMBER	English	How many grams are in one kilogram?
11	NUMBER	Arabic	كم عجلة في الدراجة العادية؟
12	NUMBER	Arabic	في أي سنة غرقت سفينة تيتانيك؟
13	NUMBER	Arabic	كم ساعة في اليوم الكامل؟
14	NUMBER	Arabic	ما ناتج قسمة 12 على 3؟
15	NUMBER	Arabic	كم درجة في الزاوية القائمة؟
16	NUMBER	Arabic	ما قيمة 20% من 200؟
17	NUMBER	Arabic	كم حرفًا في الأبجدية العربية؟
18	NUMBER	Arabic	ما قيمة 7 تربيع؟
19	NUMBER	Arabic	كم شهرًا يحتوي على 31 يومًا؟
20	NUMBER	Arabic	كم غرامًا في الكيلوغرام الواحد؟
21	LOCATION	English	Where is the Statue of Liberty located?
22	LOCATION	English	Which country is Lisbon in?
23	LOCATION	English	What is the capital city of Australia?
24	LOCATION	English	On which continent is Argentina located?
25	LOCATION	English	Where are the Pyramids of Giza located?
26	LOCATION	English	Which ocean is west of the United States?
27	LOCATION	English	Which city is famous for Big Ben?
28	LOCATION	English	In which country is the Taj Mahal located?
29	LOCATION	English	Where is Lake Victoria located?
30	LOCATION	English	Which region is known as the Middle East?
31	LOCATION	Arabic	أين يقع تمثال الحرية؟
32	LOCATION	Arabic	في أي دولة تقع لشبونة؟
33	LOCATION	Arabic	ما عاصمة أستراليا؟
34	LOCATION	Arabic	في أي قارة تقع الأرجنتين؟
35	LOCATION	Arabic	أين تقع أهرامات الجيزة؟
36	LOCATION	Arabic	ما المحيط الموجود غرب الولايات المتحدة؟
37	LOCATION	Arabic	ما المدينة المشهورة بساعة بيغ بن؟
38	LOCATION	Arabic	في أي دولة يقع تاج محل؟
39	LOCATION	Arabic	أين تقع بحيرة فيكتوريا؟
40	LOCATION	Arabic	ما المنطقة المعروفة باسم الشرق الأوسط؟
41	PERSON	English	Who wrote Pride and Prejudice?
42	PERSON	English	Who created the character Sherlock Holmes?
43	PERSON	English	Who discovered penicillin?
44	PERSON	English	Who painted The Starry Night?
45	PERSON	English	Who was the first woman to win a Nobel Prize?
46	PERSON	English	Who led India’s independence movement through nonviolent resistance?
47	PERSON	English	Who is the author of The Lord of the Rings?
48	PERSON	English	Who invented the World Wide Web?
49	PERSON	English	Who was Cleopatra?
50	PERSON	English	Who composed Moonlight Sonata?
51	PERSON	Arabic	من كتب رواية كبرياء وتحامل؟
52	PERSON	Arabic	من ابتكر شخصية شرلوك هولمز؟
53	PERSON	Arabic	من اكتشف البنسلين؟
54	PERSON	Arabic	من رسم لوحة ليلة النجوم؟
55	PERSON	Arabic	من كانت أول امرأة تفوز بجائزة نوبل؟
56	PERSON	Arabic	من قاد حركة استقلال الهند بالمقاومة السلمية؟
57	PERSON	Arabic	من هو مؤلف رواية سيد الخواتم؟
58	PERSON	Arabic	من اخترع شبكة الويب العالمية؟
59	PERSON	Arabic	من كانت كليوباترا؟
60	PERSON	Arabic	من ألّف سوناتا ضوء القمر؟
61	DESCRIPTION	English	What is gravity?
62	DESCRIPTION	English	What does recycling mean?
63	DESCRIPTION	English	What is the purpose of an operating system?
64	DESCRIPTION	English	How does the internet work?
65	DESCRIPTION	English	What is evaporation?
66	DESCRIPTION	English	What does cultural diversity mean?
67	DESCRIPTION	English	What is the role of the lungs?
68	DESCRIPTION	English	What is blockchain?
69	DESCRIPTION	English	What does supply and demand mean?
70	DESCRIPTION	English	What is the function of a search engine?
71	DESCRIPTION	Arabic	ما هي الجاذبية؟
72	DESCRIPTION	Arabic	ماذا تعني إعادة التدوير؟
73	DESCRIPTION	Arabic	ما وظيفة نظام التشغيل؟
74	DESCRIPTION	Arabic	كيف يعمل الإنترنت؟
75	DESCRIPTION	Arabic	ما هو التبخر؟
76	DESCRIPTION	Arabic	ماذا يعني التنوع الثقافي؟
77	DESCRIPTION	Arabic	ما دور الرئتين؟
78	DESCRIPTION	Arabic	ما هي تقنية البلوكشين؟
79	DESCRIPTION	Arabic	ماذا يعني العرض والطلب؟
80	DESCRIPTION	Arabic	ما وظيفة محرك البحث؟
81	ENTITY	English	What fruit is commonly used to make orange juice?
82	ENTITY	English	What tool is used to cut paper?
83	ENTITY	English	What vehicle runs on railway tracks?
84	ENTITY	English	What object is used to unlock a door?
85	ENTITY	English	What food is made from milk and often used on pizza?
86	ENTITY	English	What machine is used to wash clothes?
87	ENTITY	English	What material is commonly used to make windows?
88	ENTITY	English	What app is commonly used for video meetings?
89	ENTITY	English	What natural satellite orbits Earth?
90	ENTITY	English	What sport uses a racket and a shuttlecock?
91	ENTITY	Arabic	ما الفاكهة المستخدمة عادة لصنع عصير البرتقال؟
92	ENTITY	Arabic	ما الأداة المستخدمة لقص الورق؟
93	ENTITY	Arabic	ما المركبة التي تسير على السكك الحديدية؟
94	ENTITY	Arabic	ما الشيء المستخدم لفتح الباب؟
95	ENTITY	Arabic	ما الطعام المصنوع من الحليب ويستخدم غالبًا على البيتزا؟
96	ENTITY	Arabic	ما الآلة المستخدمة لغسل الملابس؟
97	ENTITY	Arabic	ما المادة المستخدمة عادة لصنع النوافذ؟
98	ENTITY	Arabic	ما التطبيق المستخدم عادة لاجتماعات الفيديو؟
99	ENTITY	Arabic	ما القمر الطبيعي الذي يدور حول الأرض؟
100	ENTITY	Arabic	ما الرياضة التي تستخدم مضربًا وريشة؟
101	ABBREVIATION	English	What does ATM stand for?
102	ABBREVIATION	English	What does FAQ stand for?
103	ABBREVIATION	English	What does HTTP stand for?
104	ABBREVIATION	English	What does SQL stand for?
105	ABBREVIATION	English	What does LAN stand for?
106	ABBREVIATION	English	What does VPN stand for?
107	ABBREVIATION	English	What does API stand for?
108	ABBREVIATION	English	What does SMS stand for?
109	ABBREVIATION	English	What does CEO stand for?
110	ABBREVIATION	English	What does NGO stand for?
111	ABBREVIATION	Arabic	ماذا يعني اختصار ATM؟
112	ABBREVIATION	Arabic	ماذا يعني اختصار FAQ؟
113	ABBREVIATION	Arabic	ماذا يعني اختصار HTTP؟
114	ABBREVIATION	Arabic	ماذا يعني اختصار SQL؟
115	ABBREVIATION	Arabic	ماذا يعني اختصار LAN؟
116	ABBREVIATION	Arabic	ماذا يعني اختصار VPN؟
117	ABBREVIATION	Arabic	ماذا يعني اختصار API؟
118	ABBREVIATION	Arabic	ماذا يعني اختصار SMS؟
119	ABBREVIATION	Arabic	ماذا يعني اختصار CEO؟
120	ABBREVIATION	Arabic	ماذا يعني اختصار NGO؟
"""


def load_dataset(tsv_text: str, max_samples_per_language=None) -> pd.DataFrame:
    """
    Load dataset from embedded text.

    The embedded rows use mixed spacing. This parser splits each line into:
    id, label, language, question
    while preserving normal spaces inside the question text.
    """
    rows = []

    for line in tsv_text.strip().splitlines():
        line = line.strip()

        if not line:
            continue

        if line.lower().startswith("id "):
            continue

        parts = re.split(r"\s+", line, maxsplit=3)

        if len(parts) != 4:
            raise ValueError(f"Could not parse line: {line}")

        sample_id, label, language, question = parts

        rows.append({
            "id": sample_id,
            "label": label,
            "language": language,
            "question": question.strip(),
        })

    df = pd.DataFrame(rows)

    df["language"] = df["language"].astype(str).str.strip()
    df["label"] = df["label"].astype(str).str.strip()
    df["question"] = df["question"].astype(str).str.strip()
    df["language_key"] = df["language"].str.lower()

    if max_samples_per_language is not None:
        df = (
            df.groupby("language_key", group_keys=False)
            .head(max_samples_per_language)
            .reset_index(drop=True)
        )

    return df


pilot01_df = load_dataset(PILOT01_TSV, MAX_SAMPLES_PER_LANGUAGE if QUICK_TEST else None)
pilot02_df = load_dataset(PILOT02_TSV, MAX_SAMPLES_PER_LANGUAGE if QUICK_TEST else None)
development_df = load_dataset(DEVELOPMENT_TSV, MAX_SAMPLES_PER_LANGUAGE if QUICK_TEST else None)
validation_df = load_dataset(VALIDATION_TSV, MAX_SAMPLES_PER_LANGUAGE if QUICK_TEST else None)

print("Pilot_01 distribution")
display(pilot01_df.groupby(["language", "label"]).size().reset_index(name="count"))

print("Pilot_02 distribution")
display(pilot02_df.groupby(["language", "label"]).size().reset_index(name="count"))

print("Development dataset distribution (Pilot_03--Pilot_06)")
display(development_df.groupby(["language", "label"]).size().reset_index(name="count"))

print("Unseen validation dataset distribution (Pilot_07 and cross-model)")
display(validation_df.groupby(["language", "label"]).size().reset_index(name="count"))

## 3. Prompt strategies

In [ ]:
# =========================
# Prompt guides
# =========================

LABEL_DEFINITIONS_EN = """
Use these label definitions:

NUMBER:
The expected answer is a number, date, year, quantity, measurement, percentage, or calculation result.

LOCATION:
The expected answer is a place, country, city, continent, region, river, mountain, desert, landmark, or geographical location.

PERSON:
The expected answer is a human person, fictional character, author, inventor, founder, president, artist, scientist, composer, or historical figure.

DESCRIPTION:
The expected answer requires an explanation, definition, meaning, purpose, function, process, use, or description.

ENTITY:
The expected answer is a concrete or named non-person, non-location, non-number item, such as an object, animal, planet, language, currency, software application, gas, element, metal, device, or instrument.

ABBREVIATION:
The question asks what an acronym, abbreviation, or shortened form stands for or means.
""".strip()

LABEL_DEFINITIONS_AR = """
استخدم تعريفات التصنيفات التالية:

NUMBER:
تكون الإجابة المتوقعة رقمًا أو تاريخًا أو سنة أو كمية أو قياسًا أو نسبة مئوية أو نتيجة عملية حسابية.

LOCATION:
تكون الإجابة المتوقعة مكانًا أو دولة أو مدينة أو قارة أو منطقة أو نهرًا أو جبلًا أو صحراء أو معلمًا أو موقعًا جغرافيًا.

PERSON:
تكون الإجابة المتوقعة شخصًا حقيقيًا أو شخصية خيالية أو كاتبًا أو مخترعًا أو مؤسسًا أو رئيسًا أو فنانًا أو عالمًا أو ملحنًا أو شخصية تاريخية.

DESCRIPTION:
تتطلب الإجابة شرحًا أو تعريفًا أو معنى أو هدفًا أو وظيفة أو استخدامًا أو عملية أو وصفًا.

ENTITY:
تكون الإجابة المتوقعة شيئًا محددًا أو كيانًا غير شخص وغير موقع وغير رقم، مثل جهاز أو حيوان أو كوكب أو لغة أو عملة أو تطبيق برمجي أو غاز أو عنصر أو معدن أو آلة.

ABBREVIATION:
يسأل السؤال عن معنى اختصار أو رمز مختصر أو الحروف التي يتكون منها الاختصار.
""".strip()

FEWSHOT_EN = """
Examples:
Question: How many colors are in a rainbow?
Label: NUMBER

Question: Where is the Statue of Liberty located?
Label: LOCATION

Question: Who wrote Pride and Prejudice?
Label: PERSON

Question: What is gravity?
Label: DESCRIPTION

Question: What is the function of the heart?
Label: DESCRIPTION

Question: What is a database used for?
Label: DESCRIPTION

Question: What tool is used to cut paper?
Label: ENTITY

Question: What planet is known as the Red Planet?
Label: ENTITY

Question: What does WHO stand for?
Label: ABBREVIATION
""".strip()

FEWSHOT_AR = """
أمثلة:
السؤال: كم عدد أيام شهر فبراير في السنة العادية؟
التصنيف: NUMBER

السؤال: أين تقع الأهرامات؟
التصنيف: LOCATION

السؤال: من كتب رواية البؤساء؟
التصنيف: PERSON

السؤال: ما معنى إعادة التدوير؟
التصنيف: DESCRIPTION

السؤال: ما هو تعلم الآلة؟
التصنيف: DESCRIPTION

السؤال: ما وظيفة القلب؟
التصنيف: DESCRIPTION

السؤال: ما استخدام قاعدة البيانات؟
التصنيف: DESCRIPTION

السؤال: ما الأداة المستخدمة لقص الورق؟
التصنيف: ENTITY

السؤال: ما الكوكب المعروف بالكوكب الأحمر؟
التصنيف: ENTITY

السؤال: ماذا يعني اختصار WHO؟
التصنيف: ABBREVIATION
""".strip()

DISAMBIGUATION_EN = """
Important DESCRIPTION vs ENTITY rule:
Choose DESCRIPTION when the question asks for an explanation, meaning, definition, function, purpose, use, or process.
Choose ENTITY when the question asks for the name of a specific object, animal, planet, language, currency, device, material, gas, software, element, metal, or instrument.
""".strip()

DISAMBIGUATION_AR = """
قاعدة مهمة للتمييز بين DESCRIPTION و ENTITY:
اختر DESCRIPTION إذا كان السؤال يطلب شرحًا أو تعريفًا أو معنى أو وظيفة أو استخدامًا أو هدفًا أو عملية.
اختر ENTITY إذا كان السؤال يطلب اسم شيء محدد مثل جهاز أو حيوان أو كوكب أو لغة أو عملة أو غاز أو عنصر أو معدن أو آلة.

إشارات قوية لتصنيف DESCRIPTION:
- ما هو ...؟ عندما يطلب تعريفًا أو شرحًا
- ما هي ...؟ عندما يطلب تعريفًا أو شرحًا
- ماذا يعني ...؟
- ما معنى ...؟
- ما وظيفة ...؟
- ما استخدام ...؟
- ما الهدف من ...؟
""".strip()

In [ ]:
def get_prompt_variants(strategy: str, language: str) -> list[str]:
    """Return 8 prompt variants for a given strategy and language.

    Strategies:
    - simple: label-only prompts
    - definitions: explicit label definitions
    - fewshot: definitions + examples
    - disambig: definitions + examples + DESCRIPTION/ENTITY rule
    """
    language = language.lower()

    assert strategy in {"simple", "definitions", "fewshot", "disambig"}
    assert language in {"english", "arabic"}

    if language == "english":
        simple = [
            "Classify the following question into exactly one label: NUMBER, LOCATION, PERSON, DESCRIPTION, ENTITY, ABBREVIATION. Answer with the label only.",
            "Determine the correct class for the following question using one of these labels: NUMBER, LOCATION, PERSON, DESCRIPTION, ENTITY, ABBREVIATION. Output only the label.",
            "Read the question and assign exactly one class from: NUMBER, LOCATION, PERSON, DESCRIPTION, ENTITY, ABBREVIATION. Return only the class name.",
            "Your task is to classify the question into one of the following categories: NUMBER, LOCATION, PERSON, DESCRIPTION, ENTITY, ABBREVIATION. Respond only with the category.",
            "Identify the answer type of the question using exactly one of these labels: NUMBER, LOCATION, PERSON, DESCRIPTION, ENTITY, ABBREVIATION. Do not explain.",
            "Select the best label for the question from this set: NUMBER, LOCATION, PERSON, DESCRIPTION, ENTITY, ABBREVIATION. Output the label only.",
            "Choose one class for the question among NUMBER, LOCATION, PERSON, DESCRIPTION, ENTITY, ABBREVIATION. Return only that class.",
            "Assign a single category to the question from NUMBER, LOCATION, PERSON, DESCRIPTION, ENTITY, ABBREVIATION. Answer with only the category name.",
        ]

        guide_parts = [LABEL_DEFINITIONS_EN]

        if strategy in {"fewshot", "disambig"}:
            guide_parts.append(FEWSHOT_EN)

        if strategy == "disambig":
            guide_parts.insert(1, DISAMBIGUATION_EN)

        guide = "\n\n".join(guide_parts)

        if strategy == "simple":
            return simple

        return [
            f"{guide}\n\nNow classify the following new question into exactly one label: NUMBER, LOCATION, PERSON, DESCRIPTION, ENTITY, ABBREVIATION. Answer with the label only.",
            f"{guide}\n\nUsing the definitions, examples, and disambiguation rule above when available, determine the correct answer type for the new question. Output only one label.",
            f"{guide}\n\nRead the new question carefully and assign the best matching label. Return only one label from the allowed set.",
            f"{guide}\n\nYour task is answer-type classification. Use the definitions, examples, and disambiguation rule above when available. Respond with exactly one category name only.",
            f"{guide}\n\nIdentify what type of answer the new question is asking for. Choose only one of the six labels. Do not explain.",
            f"{guide}\n\nClassify the new question into the most appropriate answer type. Output the label only.",
            f"{guide}\n\nSelect the single best class for the new question. The answer must be exactly one of: NUMBER, LOCATION, PERSON, DESCRIPTION, ENTITY, ABBREVIATION.",
            f"{guide}\n\nAssign one category to the new question based on the expected answer type. Return only the category name.",
        ]

    simple = [
        "صنّف السؤال التالي ضمن تصنيف واحد فقط من هذه التصنيفات: NUMBER, LOCATION, PERSON, DESCRIPTION, ENTITY, ABBREVIATION. أجب بالتصنيف فقط.",
        "حدّد الفئة الصحيحة للسؤال التالي باستخدام واحد فقط من التصنيفات التالية: NUMBER, LOCATION, PERSON, DESCRIPTION, ENTITY, ABBREVIATION. أخرج اسم التصنيف فقط.",
        "اقرأ السؤال ثم اختر له فئة واحدة فقط من: NUMBER, LOCATION, PERSON, DESCRIPTION, ENTITY, ABBREVIATION. أعد اسم الفئة فقط.",
        "مهمتك هي تصنيف السؤال إلى واحدة من الفئات التالية: NUMBER, LOCATION, PERSON, DESCRIPTION, ENTITY, ABBREVIATION. أجب بالفئة فقط.",
        "عرّف نوع الإجابة المطلوبة في السؤال باستخدام تصنيف واحد فقط من: NUMBER, LOCATION, PERSON, DESCRIPTION, ENTITY, ABBREVIATION. لا تشرح.",
        "اختر أفضل تصنيف للسؤال من المجموعة التالية: NUMBER, LOCATION, PERSON, DESCRIPTION, ENTITY, ABBREVIATION. أخرج اسم التصنيف فقط.",
        "اختر فئة واحدة للسؤال من بين NUMBER, LOCATION, PERSON, DESCRIPTION, ENTITY, ABBREVIATION. أعد اسم الفئة فقط.",
        "أسند للسؤال تصنيفًا واحدًا فقط من NUMBER, LOCATION, PERSON, DESCRIPTION, ENTITY, ABBREVIATION. أجب باسم التصنيف فقط.",
    ]

    guide_parts = [LABEL_DEFINITIONS_AR]

    if strategy in {"fewshot", "disambig"}:
        guide_parts.append(FEWSHOT_AR)

    if strategy == "disambig":
        guide_parts.insert(1, DISAMBIGUATION_AR)

    guide = "\n\n".join(guide_parts)

    if strategy == "simple":
        return simple

    return [
        f"{guide}\n\nالآن صنّف السؤال الجديد التالي ضمن تصنيف واحد فقط من هذه التصنيفات: NUMBER, LOCATION, PERSON, DESCRIPTION, ENTITY, ABBREVIATION. أجب بالتصنيف فقط.",
        f"{guide}\n\nباستخدام التعريفات والأمثلة وقاعدة التمييز أعلاه عند توفرها، حدّد نوع الإجابة المطلوبة في السؤال الجديد. أخرج تصنيفًا واحدًا فقط.",
        f"{guide}\n\nاقرأ السؤال الجديد بعناية واختر التصنيف الأنسب له. أعد اسم التصنيف فقط.",
        f"{guide}\n\nمهمتك هي تصنيف نوع الإجابة المطلوبة في السؤال الجديد. استخدم التعريفات السابقة وأجب بتصنيف واحد فقط.",
        f"{guide}\n\nعرّف نوع الإجابة التي يبحث عنها السؤال الجديد. اختر واحدًا فقط من التصنيفات الستة. لا تشرح.",
        f"{guide}\n\nصنّف السؤال الجديد إلى نوع الإجابة الأنسب. أخرج التصنيف فقط.",
        f"{guide}\n\nاختر أفضل فئة واحدة للسؤال الجديد. يجب أن تكون الإجابة واحدة فقط من: NUMBER, LOCATION, PERSON, DESCRIPTION, ENTITY, ABBREVIATION.",
        f"{guide}\n\nأسند للسؤال الجديد تصنيفًا واحدًا بناءً على نوع الإجابة المتوقعة. أعد اسم التصنيف فقط.",
    ]


print(get_prompt_variants("disambig", "english")[0][:1000])
print("\n--- Arabic preview ---\n")
print(get_prompt_variants("disambig", "arabic")[0][:1000])

## 4. Model loading and generation helpers

In [ ]:
if RUN_FALCON_PROMPT_PROGRESSION or RUN_CROSS_MODEL_VALIDATION:
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
else:
    AutoTokenizer = None
    AutoModelForCausalLM = None
    BitsAndBytesConfig = None


def get_hf_token():
    """Return HF token from Colab secrets or environment if available."""
    token = os.environ.get("HF_TOKEN")

    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN") or token
    except Exception:
        pass

    return token


def clear_gpu():
    global model, tokenizer

    try:
        del model
    except NameError:
        pass

    try:
        del tokenizer
    except NameError:
        pass

    gc.collect()

    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("GPU memory cleared.")


def load_quantized_model(model_id: str):
    """Load a causal LM in 4-bit quantization for Colab T4."""

    if not (RUN_FALCON_PROMPT_PROGRESSION or RUN_CROSS_MODEL_VALIDATION):
        raise RuntimeError("HF model loading was requested, but HF run modes are disabled.")

    token = get_hf_token()

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

    print("Loading tokenizer:", model_id)

    tok = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True,
        token=token,
    )

    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    print("Loading model:", model_id)

    mdl = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        token=token,
    )

    mdl.eval()

    print("Model loaded:", model_id)

    return tok, mdl


def build_chat_prompt(tokenizer, instruction: str, question: str) -> str:
    user_msg = f"{instruction}\n\nNew question:\n{question}\n\nLabel:"

    messages = [
        {
            "role": "system",
            "content": (
                "You are a strict question-type classifier. "
                "You must output only one label from the allowed label set."
            ),
        },
        {
            "role": "user",
            "content": user_msg
        },
    ]

    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        return (
            "System: You are a strict question-type classifier. Output only one label.\n"
            f"User: {user_msg}\n"
            "Assistant:"
        )


def generate_label(tokenizer, model, instruction: str, question: str) -> str:
    prompt = build_chat_prompt(tokenizer, instruction, question)
    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=12,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    output_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    return output_text


def normalize_label(raw_output: str) -> str:
    text = str(raw_output).strip().upper()
    text = re.sub(r"[^A-Z_ ]", " ", text)

    for label in LABELS:
        if text == label:
            return label

    for label in LABELS:
        if label in text:
            return label

    return "N/A"

## 5. Metrics and experiment runner

In [ ]:
def normalized_entropy(preds: list[str]) -> float:
    counts = Counter(preds)
    total = sum(counts.values())
    if total == 0:
        return 0.0
    probs = np.array([count / total for count in counts.values()])
    entropy = -np.sum(probs * np.log(probs))
    max_entropy = np.log(len(LABELS) + 1)  # include possible N/A
    return float(entropy / max_entropy)

def total_variation_distance(p: dict, q: dict) -> float:
    keys = set(p.keys()) | set(q.keys())
    return 0.5 * sum(abs(p.get(k, 0.0) - q.get(k, 0.0)) for k in keys)

def prediction_distribution(preds: list[str]) -> dict:
    counts = Counter(preds)
    total = sum(counts.values())
    if total == 0:
        return {}
    return {k: v / total for k, v in counts.items()}

def compute_consistency(distributions: dict, gold_labels: dict) -> tuple[float, dict]:
    grouped = defaultdict(list)
    for sample_id, dist in distributions.items():
        grouped[gold_labels[sample_id]].append(dist)

    class_consistency = {}
    for label, dists in grouped.items():
        scores = []
        for i in range(len(dists)):
            for j in range(i + 1, len(dists)):
                score = 1.0 - total_variation_distance(dists[i], dists[j])
                scores.append(score)
        class_consistency[label] = float(np.mean(scores)) if scores else None

    valid_scores = [v for v in class_consistency.values() if v is not None]
    overall = float(np.mean(valid_scores)) if valid_scores else 0.0
    return overall, class_consistency

def type_token_ratio(texts: list[str]) -> float:
    tokens = []
    for text in texts:
        tokens.extend(str(text).split())
    return len(set(tokens)) / len(tokens) if tokens else 0.0

def chars_per_word(texts: list[str]) -> float:
    values = []
    for text in texts:
        words = str(text).split()
        if words:
            values.append(len(str(text)) / len(words))
    return float(np.mean(values)) if values else 0.0


def run_experiment(
    *,
    tokenizer,
    model,
    model_id: str,
    experiment_id: str,
    output_prefix: str,
    dataset_df: pd.DataFrame,
    strategy: str,
    results_dir: Path = RESULTS_DIR,
):
    """Run one full experiment and save prompt-level, sample-level, metrics, and error CSV files."""
    exp_dir = results_dir / output_prefix
    exp_dir.mkdir(parents=True, exist_ok=True)

    raw_rows = []
    metric_rows = []
    sample_rows = []

    for language_key, samples_df in dataset_df.groupby("language_key"):
        prompts = get_prompt_variants(strategy, language_key)

        sample_to_preds = defaultdict(list)
        sample_to_raw_outputs = defaultdict(list)
        sample_to_gold = {}

        records = samples_df.to_dict("records")

        for sample in tqdm(records, desc=f"{experiment_id} | {language_key}"):
            sample_id = str(sample["id"])
            sample_to_gold[sample_id] = sample["label"]

            for prompt_id, instruction in enumerate(prompts):
                raw_output = generate_label(tokenizer, model, instruction, sample["question"])
                pred = normalize_label(raw_output)

                sample_to_preds[sample_id].append(pred)
                sample_to_raw_outputs[sample_id].append(raw_output)

                raw_rows.append({
                    "model": model_id,
                    "experiment_id": experiment_id,
                    "strategy": strategy,
                    "language": language_key,
                    "sample_id": sample_id,
                    "text": sample["question"],
                    "gold": sample["label"],
                    "prompt_id": prompt_id,
                    "raw_output": raw_output,
                    "prediction": pred,
                    "is_correct_prompt_level": pred == sample["label"],
                })

        majority_predictions = []
        gold_labels = []
        sensitivities = []
        distributions = {}

        for sample in records:
            sample_id = str(sample["id"])
            preds = sample_to_preds[sample_id]
            dist = prediction_distribution(preds)
            majority_pred = Counter(preds).most_common(1)[0][0]

            majority_predictions.append(majority_pred)
            gold_labels.append(sample["label"])

            sensitivity = normalized_entropy(preds)
            sensitivities.append(sensitivity)
            distributions[sample_id] = dist

            sample_rows.append({
                "model": model_id,
                "experiment_id": experiment_id,
                "strategy": strategy,
                "language": language_key,
                "sample_id": sample_id,
                "text": sample["question"],
                "gold": sample["label"],
                "majority_prediction": majority_pred,
                "is_correct_majority": majority_pred == sample["label"],
                "prompt_sensitivity": sensitivity,
                "prediction_counts": json.dumps(dict(Counter(preds)), ensure_ascii=False),
                "prediction_distribution": json.dumps(dist, ensure_ascii=False),
                "raw_outputs": json.dumps(sample_to_raw_outputs[sample_id], ensure_ascii=False),
            })

        acc = accuracy_score(gold_labels, majority_predictions)

        macro_f1 = f1_score(
            gold_labels,
            majority_predictions,
            average="macro",
            labels=LABELS,
            zero_division=0,
        )

        consistency, consistency_by_class = compute_consistency(distributions, sample_to_gold)
        texts = [sample["question"] for sample in records]

        metric_rows.append({
            "model": model_id,
            "experiment_id": experiment_id,
            "strategy": strategy,
            "language": language_key,
            "n_samples": len(records),
            "n_prompts": len(prompts),
            "n_generations": len(records) * len(prompts),
            "accuracy": acc,
            "macro_f1": macro_f1,
            "avg_sensitivity": float(np.mean(sensitivities)),
            "avg_consistency": consistency,
            "type_token_ratio": type_token_ratio(texts),
            "chars_per_word": chars_per_word(texts),
            "consistency_by_class": json.dumps(consistency_by_class, ensure_ascii=False),
        })

    raw_df = pd.DataFrame(raw_rows)
    metrics_df = pd.DataFrame(metric_rows)
    sample_df = pd.DataFrame(sample_rows)
    error_df = sample_df[sample_df["is_correct_majority"] == False].copy()
    prompt_error_df = raw_df[raw_df["is_correct_prompt_level"] == False].copy()

    class_level_df = (
        sample_df
        .groupby(["experiment_id", "model", "strategy", "language", "gold"])
        .agg(
            n_samples=("sample_id", "count"),
            n_correct=("is_correct_majority", "sum"),
            accuracy=("is_correct_majority", "mean"),
            avg_prompt_sensitivity=("prompt_sensitivity", "mean"),
        )
        .reset_index()
        .rename(columns={"gold": "label"})
    )

    raw_df.to_csv(exp_dir / f"raw_predictions_{output_prefix}.csv", index=False, encoding="utf-8")
    metrics_df.to_csv(exp_dir / f"metrics_{output_prefix}.csv", index=False, encoding="utf-8")
    sample_df.to_csv(exp_dir / f"sample_level_results_{output_prefix}.csv", index=False, encoding="utf-8")
    error_df.to_csv(exp_dir / f"error_analysis_{output_prefix}.csv", index=False, encoding="utf-8")
    prompt_error_df.to_csv(exp_dir / f"prompt_level_errors_{output_prefix}.csv", index=False, encoding="utf-8")
    class_level_df.to_csv(exp_dir / f"class_level_results_{output_prefix}.csv", index=False, encoding="utf-8")

    print(f"\nFinished {experiment_id}. Files saved in {exp_dir}")
    display(metrics_df)
    display(class_level_df)
    display(error_df)

    return {
        "raw": raw_df,
        "metrics": metrics_df,
        "sample": sample_df,
        "errors": error_df,
        "prompt_errors": prompt_error_df,
        "class_level": class_level_df,
        "output_dir": exp_dir,
    }

## 6. Falcon experiment sequence: Pilot_01 to Pilot_07

Pilot_01 and Pilot_02 are included for reproducibility and research-traceability. They are preliminary debugging/development pilots, while Pilot_03--Pilot_07 form the main controlled prompt-design sequence reported in the paper.


In [ ]:
# This reproduces the full Falcon3 experiment sequence.
# Pilot_01 and Pilot_02 are preliminary pipeline/dataset pilots.
# Pilot_03 to Pilot_07 are the main prompt-design experiments.

falcon_outputs = {}

if RUN_FALCON_PROMPT_PROGRESSION:
    clear_gpu()
    tokenizer, model = load_quantized_model(FALCON_MODEL_ID)

    experiments = [
        {
            "experiment_id": "Pilot_01_Tiny_Starter_Dataset",
            "output_prefix": "hf_pilot01",
            "dataset": pilot01_df,
            "strategy": "simple",
        },
        {
            "experiment_id": "Pilot_02_Expanded_PreCleaning_Dataset",
            "output_prefix": "hf_pilot02",
            "dataset": pilot02_df,
            "strategy": "simple",
        },
        {
            "experiment_id": "Pilot_03_Simple_Prompts",
            "output_prefix": "hf_pilot03",
            "dataset": development_df,
            "strategy": "simple",
        },
        {
            "experiment_id": "Pilot_04_Definition_Prompts",
            "output_prefix": "hf_pilot04",
            "dataset": development_df,
            "strategy": "definitions",
        },
        {
            "experiment_id": "Pilot_05_Definitions_FewShot",
            "output_prefix": "hf_pilot05",
            "dataset": development_df,
            "strategy": "fewshot",
        },
        {
            "experiment_id": "Pilot_06_Description_Entity_Disambiguation",
            "output_prefix": "hf_pilot06",
            "dataset": development_df,
            "strategy": "disambig",
        },
        {
            "experiment_id": "Pilot_07_Unseen_Validation",
            "output_prefix": "hf_pilot07",
            "dataset": validation_df,
            "strategy": "disambig",
        },
    ]

    for exp in experiments:
        falcon_outputs[exp["output_prefix"]] = run_experiment(
            tokenizer=tokenizer,
            model=model,
            model_id=FALCON_MODEL_ID,
            experiment_id=exp["experiment_id"],
            output_prefix=exp["output_prefix"],
            dataset_df=exp["dataset"],
            strategy=exp["strategy"],
        )

    clear_gpu()
else:
    print("Skipping Falcon prompt progression.")

## 7. Cross-model validation with the frozen final prompt

In [ ]:
# This reproduces the final 3-model cross-model validation on the unseen validation dataset.
# The final prompt strategy is frozen: definitions + examples + DESCRIPTION/ENTITY rule.

cross_model_outputs = {}

if RUN_CROSS_MODEL_VALIDATION:
    for short_name, model_id, model_type in CROSS_MODEL_RUN_ORDER:
        output_prefix = f"crossmodel_{short_name}"
        experiment_id = f"CrossModel_{short_name}_Pilot07"

        clear_gpu()
        tokenizer, model = load_quantized_model(model_id)

        cross_model_outputs[short_name] = run_experiment(
            tokenizer=tokenizer,
            model=model,
            model_id=model_id,
            experiment_id=experiment_id,
            output_prefix=output_prefix,
            dataset_df=validation_df,
            strategy="disambig",
        )

        clear_gpu()
else:
    print("Skipping cross-model validation.")

## 8. Gemini closed-model validation

This section evaluates the frozen final prompt on a closed proprietary model: `gemini-2.5-flash`.

Before running this section:

1. Open the Colab **Secrets** tab.
2. Add a secret named `GEMINI_API_KEY`.
3. Paste your Gemini API key as the value.
4. Enable notebook access for the secret.

This section uses the Pilot_07 validation dataset, eight compact variants of the frozen final prompt, the same majority-vote method, and the same metrics. For Gemini 2.5 Flash, thinking is disabled with `thinking_budget=0` and `max_output_tokens=64` to avoid empty MAX_TOKENS responses.

It saves outputs under:

`results/crossmodel_gemini_2_5_flash/`

In [ ]:
# =========================
# GEMINI CELL 1 — Setup client and response extraction
# =========================

import os
import time

genai = None
types = None

EXPERIMENT_ID_GEMINI = "CrossModel_Gemini_2_5_Flash_Pilot07"
OUTPUT_PREFIX_GEMINI = "crossmodel_gemini_2_5_flash"


def get_gemini_api_key():
    """Return Gemini API key from Colab Secrets or environment."""
    key = os.environ.get("GEMINI_API_KEY")

    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY") or key
    except Exception:
        pass

    return key


def extract_gemini_text(response) -> str:
    """
    Robustly extract text from a Gemini response.
    This handles cases where response.text is None.
    """
    if hasattr(response, "text") and response.text:
        return response.text.strip()

    try:
        candidates = getattr(response, "candidates", None)

        if candidates:
            parts = candidates[0].content.parts
            texts = []

            for part in parts:
                if hasattr(part, "text") and part.text:
                    texts.append(part.text)

            if texts:
                return " ".join(texts).strip()

    except Exception:
        pass

    return ""


gemini_client = None

if RUN_GEMINI_CLOSED_MODEL_VALIDATION:
    try:
        from google import genai
        from google.genai import types
    except Exception as exc:
        raise RuntimeError(
            "google-genai is required for Gemini reruns. Install requirements.txt or run the setup cell."
        ) from exc

    api_key = get_gemini_api_key()

    if not api_key:
        raise RuntimeError(
            "GEMINI_API_KEY not found. Add it in Colab Secrets and enable notebook access."
        )

    gemini_client = genai.Client(api_key=api_key)

    print("Gemini client configured.")
    print("Model:", GEMINI_MODEL_ID)

else:
    print("Skipping Gemini client setup because RUN_GEMINI_CLOSED_MODEL_VALIDATION=False.")


In [ ]:
# =========================
# GEMINI CELL 2 — Quick API test
# =========================

if RUN_GEMINI_CLOSED_MODEL_VALIDATION:
    test_response = gemini_client.models.generate_content(
        model=GEMINI_MODEL_ID,
        contents=(
            "Output exactly one word: ABBREVIATION"
        ),
        config=types.GenerateContentConfig(
            temperature=0.0,
            max_output_tokens=64,
            thinking_config=types.ThinkingConfig(thinking_budget=0),
        ),
    )

    extracted_text = extract_gemini_text(test_response)

    print("Finish reason:", test_response.candidates[0].finish_reason)
    print("Extracted text:")
    print(extracted_text)

    if not extracted_text:
        print("\nWARNING: Empty extracted text. Raw response object:")
        print(test_response)

else:
    print("Skipping Gemini quick test.")

# Remove accidental header row from every loaded dataset

def clean_loaded_dataset(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["id"] = df["id"].astype(str).str.strip()
    df["label"] = df["label"].astype(str).str.strip()
    df["language"] = df["language"].astype(str).str.strip()
    df["question"] = df["question"].astype(str).str.strip()

    df = df[
        (df["id"].str.lower() != "id")
        & (df["label"].str.lower() != "label")
        & (df["language"].str.lower() != "language")
        & (df["question"].str.lower() != "question")
    ].copy()

    df["language_key"] = df["language"].str.lower()

    return df.reset_index(drop=True)


pilot01_df = clean_loaded_dataset(pilot01_df)
pilot02_df = clean_loaded_dataset(pilot02_df)
development_df = clean_loaded_dataset(development_df)
validation_df = clean_loaded_dataset(validation_df)

print("Validation rows:", len(validation_df))
print(validation_df["language_key"].value_counts())
display(validation_df.head())


In [ ]:
# =========================
# GEMINI CELL 3 — Compact Gemini generation helper
# =========================

import time

COMPACT_GUIDE_EN = """
Labels:
NUMBER = number/date/year/quantity/measurement/calculation.
LOCATION = place/country/city/continent/region/landmark/geographic location.
PERSON = human person, fictional character, author, inventor, artist, scientist, composer, historical figure.
DESCRIPTION = explanation, definition, meaning, purpose, function, use, process, or description.
ENTITY = concrete or named non-person, non-location, non-number item: object, animal, planet, language, currency, software, gas, element, metal, device, instrument.
ABBREVIATION = asks what an acronym/abbreviation stands for.

Rule:
Use DESCRIPTION for explanation/meaning/function/use/process.
Use ENTITY for a specific concrete or named thing.

Examples:
How many colors are in a rainbow? -> NUMBER
Where is the Statue of Liberty located? -> LOCATION
Who wrote Pride and Prejudice? -> PERSON
What is gravity? -> DESCRIPTION
What tool is used to cut paper? -> ENTITY
What does WHO stand for? -> ABBREVIATION
""".strip()

COMPACT_GUIDE_AR = """
Labels:
NUMBER = رقم/تاريخ/سنة/كمية/قياس/عملية حسابية.
LOCATION = مكان/دولة/مدينة/قارة/منطقة/معلم/موقع جغرافي.
PERSON = شخص حقيقي أو خيالي، كاتب، مخترع، فنان، عالم، ملحن، شخصية تاريخية.
DESCRIPTION = شرح/تعريف/معنى/هدف/وظيفة/استخدام/عملية/وصف.
ENTITY = شيء محدد غير شخص وغير مكان وغير رقم: جهاز، حيوان، كوكب، لغة، عملة، تطبيق، غاز، عنصر، معدن، آلة.
ABBREVIATION = سؤال عن معنى اختصار.

Rule:
اختر DESCRIPTION إذا كان السؤال يطلب شرحًا أو معنى أو وظيفة أو استخدامًا أو عملية.
اختر ENTITY إذا كان السؤال يطلب اسم شيء محدد.

Examples:
كم عدد أيام فبراير؟ -> NUMBER
أين تقع الأهرامات؟ -> LOCATION
من كتب رواية البؤساء؟ -> PERSON
ما معنى إعادة التدوير؟ -> DESCRIPTION
ما الأداة المستخدمة لقص الورق؟ -> ENTITY
ماذا يعني اختصار WHO؟ -> ABBREVIATION
""".strip()


def build_compact_gemini_prompt(question: str, language_key: str) -> str:
    guide = COMPACT_GUIDE_AR if language_key == "arabic" else COMPACT_GUIDE_EN

    return f"""
You are a strict classifier.
Return exactly one label from:
NUMBER, LOCATION, PERSON, DESCRIPTION, ENTITY, ABBREVIATION

{guide}

Question:
{question}

Label:
""".strip()


def generate_label_gemini(question: str, language_key: str, sleep_seconds: float = 0.4) -> str:
    """
    Gemini-only compact classification call.
    Uses the same final strategy but compressed to avoid MAX_TOKENS.
    """

    full_prompt = build_compact_gemini_prompt(question, language_key)

    try:
        response = gemini_client.models.generate_content(
            model=GEMINI_MODEL_ID,
            contents=full_prompt,
            config=types.GenerateContentConfig(
                temperature=0.0,
                max_output_tokens=256,
                thinking_config=types.ThinkingConfig(thinking_budget=0),
            ),
        )

        time.sleep(sleep_seconds)

        extracted_text = extract_gemini_text(response)

        if extracted_text:
            return extracted_text

        print("Gemini returned empty text. Finish reason:", response.candidates[0].finish_reason)
        print("Raw response:")
        print(response)
        return "N/A"

    except Exception as e:
        print("Gemini API error:", e)
        time.sleep(3)
        return "N/A"

In [ ]:
# =========================
# GEMINI CELL 4 — Smoke test on validation data
# =========================

if RUN_GEMINI_CLOSED_MODEL_VALIDATION:
    validation_df["language_key"] = (
        validation_df["language"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    validation_df = validation_df[
        validation_df["language_key"].isin(["english", "arabic"])
    ].reset_index(drop=True)

    print("Validation rows:", len(validation_df))
    print(validation_df["language_key"].value_counts())

    smoke_samples = validation_df.groupby("language_key", group_keys=False).head(1)

    for _, sample in smoke_samples.iterrows():
        language_key = str(sample["language_key"]).strip().lower()

        raw_output = generate_label_gemini(
            question=sample["question"],
            language_key=language_key,
            sleep_seconds=0.4,
        )

        pred = normalize_label(raw_output)

        print("Language:", language_key)
        print("Question:", sample["question"])
        print("Gold:", sample["label"])
        print("Raw Gemini output:", raw_output)
        print("Normalized prediction:", pred)
        print("-" * 60)

else:
    print("Skipping Gemini smoke test.")

In [ ]:
# =========================
# GEMINI CELL 5 — Full Gemini closed-model validation
# =========================

def run_gemini_experiment(
    *,
    model_id: str = GEMINI_MODEL_ID,
    experiment_id: str = EXPERIMENT_ID_GEMINI,
    output_prefix: str = OUTPUT_PREFIX_GEMINI,
    dataset_df: pd.DataFrame = validation_df,
    results_dir: Path = RESULTS_DIR,
):
    """
    Run Gemini closed-model validation.

    Gemini uses 8 compact prompt variants of the frozen final strategy.
    The prompts remain short while preserving the label definitions, examples,
    and DESCRIPTION/ENTITY rule.
    """

    exp_dir = results_dir / output_prefix
    exp_dir.mkdir(parents=True, exist_ok=True)

    dataset_df = dataset_df.copy()
    dataset_df["language_key"] = (
        dataset_df["language"]
        .astype(str)
        .str.strip()
        .str.lower()
    )
    dataset_df = dataset_df[
        dataset_df["language_key"].isin(["english", "arabic"])
    ].reset_index(drop=True)

    raw_rows = []
    metric_rows = []
    sample_rows = []

    n_prompt_variants = 8

    for language_key, samples_df in dataset_df.groupby("language_key"):
        sample_to_preds = defaultdict(list)
        sample_to_raw_outputs = defaultdict(list)
        sample_to_gold = {}

        records = samples_df.to_dict("records")

        for sample in tqdm(records, desc=f"{experiment_id} | {language_key}"):
            sample_id = str(sample["id"])
            sample_to_gold[sample_id] = sample["label"]

            for prompt_id in range(n_prompt_variants):
                raw_output = generate_label_gemini(
                    question=sample["question"],
                    language_key=language_key,
                    sleep_seconds=0.4,
                )

                pred = normalize_label(raw_output)

                sample_to_preds[sample_id].append(pred)
                sample_to_raw_outputs[sample_id].append(raw_output)

                raw_rows.append({
                    "model": model_id,
                    "experiment_id": experiment_id,
                    "strategy": "compact_disambig",
                    "language": language_key,
                    "sample_id": sample_id,
                    "text": sample["question"],
                    "gold": sample["label"],
                    "prompt_id": prompt_id,
                    "raw_output": raw_output,
                    "prediction": pred,
                    "is_correct_prompt_level": pred == sample["label"],
                })

        majority_predictions = []
        gold_labels = []
        sensitivities = []
        distributions = {}

        for sample in records:
            sample_id = str(sample["id"])
            preds = sample_to_preds[sample_id]
            dist = prediction_distribution(preds)

            majority_pred = Counter(preds).most_common(1)[0][0]

            majority_predictions.append(majority_pred)
            gold_labels.append(sample["label"])

            sensitivity = normalized_entropy(preds)
            sensitivities.append(sensitivity)

            distributions[sample_id] = dist

            sample_rows.append({
                "model": model_id,
                "experiment_id": experiment_id,
                "strategy": "compact_disambig",
                "language": language_key,
                "sample_id": sample_id,
                "text": sample["question"],
                "gold": sample["label"],
                "majority_prediction": majority_pred,
                "is_correct_majority": majority_pred == sample["label"],
                "prompt_sensitivity": sensitivity,
                "prediction_counts": json.dumps(dict(Counter(preds)), ensure_ascii=False),
                "prediction_distribution": json.dumps(dist, ensure_ascii=False),
                "raw_outputs": json.dumps(sample_to_raw_outputs[sample_id], ensure_ascii=False),
            })

        acc = accuracy_score(gold_labels, majority_predictions)

        macro_f1 = f1_score(
            gold_labels,
            majority_predictions,
            average="macro",
            labels=LABELS,
            zero_division=0,
        )

        consistency, consistency_by_class = compute_consistency(
            distributions,
            sample_to_gold,
        )

        texts = [sample["question"] for sample in records]

        metric_rows.append({
            "model": model_id,
            "experiment_id": experiment_id,
            "strategy": "compact_disambig",
            "language": language_key,
            "n_samples": len(records),
            "n_prompts": n_prompt_variants,
            "n_generations": len(records) * n_prompt_variants,
            "accuracy": acc,
            "macro_f1": macro_f1,
            "avg_sensitivity": float(np.mean(sensitivities)),
            "avg_consistency": consistency,
            "type_token_ratio": type_token_ratio(texts),
            "chars_per_word": chars_per_word(texts),
            "consistency_by_class": json.dumps(consistency_by_class, ensure_ascii=False),
        })

    raw_df = pd.DataFrame(raw_rows)
    metrics_df = pd.DataFrame(metric_rows)
    sample_df = pd.DataFrame(sample_rows)

    error_df = sample_df[sample_df["is_correct_majority"] == False].copy()
    prompt_error_df = raw_df[raw_df["is_correct_prompt_level"] == False].copy()

    class_level_df = (
        sample_df
        .groupby(["experiment_id", "model", "strategy", "language", "gold"])
        .agg(
            n_samples=("sample_id", "count"),
            n_correct=("is_correct_majority", "sum"),
            accuracy=("is_correct_majority", "mean"),
            avg_prompt_sensitivity=("prompt_sensitivity", "mean"),
        )
        .reset_index()
        .rename(columns={"gold": "label"})
    )

    raw_df.to_csv(exp_dir / f"raw_predictions_{output_prefix}.csv", index=False, encoding="utf-8")
    metrics_df.to_csv(exp_dir / f"metrics_{output_prefix}.csv", index=False, encoding="utf-8")
    sample_df.to_csv(exp_dir / f"sample_level_results_{output_prefix}.csv", index=False, encoding="utf-8")
    error_df.to_csv(exp_dir / f"error_analysis_{output_prefix}.csv", index=False, encoding="utf-8")
    prompt_error_df.to_csv(exp_dir / f"prompt_level_errors_{output_prefix}.csv", index=False, encoding="utf-8")
    class_level_df.to_csv(exp_dir / f"class_level_results_{output_prefix}.csv", index=False, encoding="utf-8")

    print(f"\nFinished {experiment_id}. Files saved in {exp_dir}")

    print("\nMetrics:")
    display(metrics_df)

    print("\nClass-level results:")
    display(class_level_df)

    print("\nMajority-vote errors:")
    display(error_df)

    return {
        "raw": raw_df,
        "metrics": metrics_df,
        "sample": sample_df,
        "errors": error_df,
        "prompt_errors": prompt_error_df,
        "class_level": class_level_df,
        "output_dir": exp_dir,
    }


gemini_outputs = {}

if RUN_GEMINI_CLOSED_MODEL_VALIDATION:
    gemini_outputs["gemini_2_5_flash"] = run_gemini_experiment()

else:
    print("Skipping Gemini closed-model validation.")

## 9. Build final summary tables

This section collects all available result CSV files and builds final summary tables.

It supports both:

1. Notebook-generated nested folders, such as `results/crossmodel_gemini_2_5_flash/`
2. Repository-style folders, such as `results/metrics/`, `results/class_level/`, and `results/error_analysis/`

In [ ]:
def collect_csv(patterns) -> pd.DataFrame:
    """Collect CSVs from one or more glob patterns under RESULTS_DIR."""

    if isinstance(patterns, str):
        patterns = [patterns]

    files = []

    for pattern in patterns:
        files.extend(sorted(RESULTS_DIR.glob(pattern)))

    seen = set()
    unique_files = []

    for file in files:
        if file not in seen:
            unique_files.append(file)
            seen.add(file)

    frames = []

    for file in unique_files:
        try:
            frames.append(pd.read_csv(file))
        except Exception as e:
            print("Could not read", file, e)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


all_metrics = collect_csv(["*/metrics_*.csv", "metrics/metrics_*.csv"])
all_class_level = collect_csv(["*/class_level_results_*.csv", "class_level/class_level_results_*.csv"])
all_errors = collect_csv(["*/error_analysis_*.csv", "error_analysis/error_analysis_*.csv"])

summary_dir = RESULTS_DIR / "summary_tables"
summary_dir.mkdir(exist_ok=True, parents=True)

if all_metrics.empty:
    print("No metrics CSV files found yet. Run at least one experiment before building summary tables.")

else:
    prelim_ids = [
        "Pilot_01_Tiny_Starter_Dataset",
        "Pilot_02_Expanded_PreCleaning_Dataset",
    ]

    table0 = all_metrics[all_metrics["experiment_id"].isin(prelim_ids)].copy()
    table0.to_csv(summary_dir / "table0_preliminary_pilots.csv", index=False)

    dataset_distribution = (
        validation_df
        .groupby(["language", "label"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=LABELS)
    )

    dataset_distribution["Total"] = dataset_distribution.sum(axis=1)
    dataset_distribution.loc["Total"] = dataset_distribution.sum(axis=0)
    dataset_distribution.to_csv(summary_dir / "table1_dataset_distribution.csv")

    falcon_prompt_ids = [
        "Pilot_01_Tiny_Starter_Dataset",
        "Pilot_02_Expanded_PreCleaning_Dataset",
        "Pilot_03_Simple_Prompts",
        "Pilot_04_Definition_Prompts",
        "Pilot_05_Definitions_FewShot",
        "Pilot_06_Description_Entity_Disambiguation",
        "Pilot_07_Unseen_Validation",
    ]

    table2 = all_metrics[all_metrics["experiment_id"].isin(falcon_prompt_ids)].copy()

    if not table2.empty:
        table2_wide = table2.pivot_table(
            index=["experiment_id", "strategy"],
            columns="language",
            values="accuracy",
            aggfunc="first",
        ).reset_index()

        table2_wide.to_csv(summary_dir / "table2_falcon_prompt_progression.csv", index=False)
        table2.to_csv(summary_dir / "table2_falcon_full_sequence_long.csv", index=False)

    table3 = all_metrics[all_metrics["experiment_id"].eq("Pilot_07_Unseen_Validation")].copy()
    table3.to_csv(summary_dir / "table3_pilot07_validation_metrics.csv", index=False)

    table4 = all_class_level[all_class_level["experiment_id"].eq("Pilot_07_Unseen_Validation")].copy()
    table4.to_csv(summary_dir / "table4_pilot07_class_level.csv", index=False)

    cross_model_ids = [f"CrossModel_{short_name}_Pilot07" for short_name, _, _ in CROSS_MODEL_RUN_ORDER]
    cross_model_ids.append("CrossModel_Gemini_2_5_Flash_Pilot07")

    table5 = all_metrics[all_metrics["experiment_id"].isin(cross_model_ids)].copy()
    table5.to_csv(summary_dir / "table5_cross_model_validation.csv", index=False)

    table6 = all_class_level[all_class_level["experiment_id"].isin(cross_model_ids)].copy()
    table6.to_csv(summary_dir / "table6_cross_model_class_level.csv", index=False)

    print("Summary tables saved to:", summary_dir)

    print("\nAll metrics:")
    display(all_metrics)

    print("\nCross-model metrics:")
    display(table5)

    print("\nAll errors:")
    display(all_errors)

## 10. Zip results for submission

This creates one downloadable zip file containing all generated CSV outputs.

In [ ]:
zip_base = Path("results_package")
zip_file = zip_base.with_suffix(".zip")

if zip_file.exists():
    zip_file.unlink()

shutil.make_archive(str(zip_base), "zip", RESULTS_DIR)

print("Created:", zip_file.resolve())

try:
    from google.colab import files
    files.download(str(zip_file))

except Exception as e:
    print("Download skipped:", e)